# IST Scenario 4 — Recognising a stroke: Face, Arm, Speech, Time

_A quick, visual tour of 19,435 stroke patients, for a mixed audience. Level: introduction, no statistics needed. About 40 minutes._

**What you will do.** Public campaigns teach everyone to recognise a stroke with four letters,
FAST. IST recorded, for every patient, the signs a doctor found when the patient entered the
trial. You will see what a stroke looks like in 19,435 real patients, check how many of their
strokes FAST would have recognised, see how quickly they reached the trial, and see what had
happened to them six months later. Each step is one or two calls to the MIP and one chart.

**Stroke in one minute.** A stroke is a "brain attack". The blood supply to part of the brain
is cut off by a clot (an *ischaemic* stroke, about 9 in 10 IST patients) or a blood vessel
bursts (a *haemorrhagic* stroke). Without blood, brain cells die within minutes. What the
patient notices depends on where in the brain it happens. Each half of the brain controls the
opposite half of the body, so a stroke in the left half weakens the right side of the face,
the right arm and the right leg. Language sits in the left half in most people. The back of
the brain and the brainstem control balance, coordination and vision.

**FAST** means **F**ace drooping, **A**rm weakness, **S**peech difficulty: **T**ime to call the
emergency number. **BE-FAST** adds **B**alance (sudden loss of balance or coordination) and
**E**yes (sudden loss of vision).

**Before you start.** IST randomised 19,435 patients between 1991 and 1996. Nobody received
thrombolysis or thrombectomy, one third were randomised before a CT scan, and "dependent" was
measured by questionnaire, not by the modified Rankin Scale. The *conclusions* of the trial still
hold and are current practice; the *absolute percentages* describe 1990s stroke care and must not
be quoted to patients.

**How this notebook works.** Every analysis is a *federated* call: the code selects variables,
defines a cohort with filters and asks the MIP for aggregates (counts, means, a test, a model).
No patient row ever reaches this notebook. Run the cells in order; each step says what to look
for and what result to expect.

## 1. Connect to the MIP and open the IST pathology

In [ ]:
import mip
from mip.filters import F

client = mip.Client.from_env()          # connection configured by the MIP portal
catalog = client.catalog()

try:
    dm = catalog.data_model("IST")      # add version="1.0" if the catalogue holds several versions
except LookupError:
    display(catalog.summaries())        # the pathology label differs: pick it from this list
    raise
ist = dm.datasets.list()[0]             # the IST pathology holds a single dataset
display(dm)
display(ist)

## 2. Toolbox

In [ ]:
# ---- Toolbox: small helpers that turn MIP results into readable tables and charts. ----
# Run this cell once. You do not need to read it to follow the scenario; come back to it
# when you want to see how a result payload is unpacked.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from mip.preprocessing import MissingValuesHandler


def labels_of(variable):
    """{code: label} of a categorical variable, whatever format the catalogue uses."""
    raw = variable._data.get("enumerations") or {}
    if isinstance(raw, dict):
        return {str(k): str(v) for k, v in raw.items()}
    out = {}
    for item in raw:
        if isinstance(item, dict):
            out[str(item.get("code"))] = str(item.get("label"))
        else:
            out[str(item)] = str(item)
    return out


def analysis(variables, filters=None, drop_missing=False):
    """A Pipeline on the IST dataset with the given variables and cohort filter.
    Tests and models need drop_missing=True: the platform requires an explicit missing-values
    step and analyses complete cases only. describe() keeps it off so it can count missing values."""
    handler = MissingValuesHandler(strategies={v: "drop" for v in variables}) if drop_missing else None
    return mip.Pipeline(analysis_set=dm.select(datasets=[ist], variables=variables), filters=filters,
                        handle_missing=handler)


def rows_for(result, variable):
    """The pooled 'describe' row of one variable (the platform adds one row per dataset as well)."""
    return [r for r in result.raw.get("featurewise", [])
            if r.get("variable") == variable._code and r.get("dataset") == "all datasets"]


def numeric_summary(result, variables):
    """Mean, SD, median and range of numeric variables from a describe result."""
    rows = []
    for v in variables:
        for r in rows_for(result, v):
            d = r.get("data") or {}
            rows.append({"variable": v.label, "n": d.get("num_dtps"), "missing": d.get("num_na"),
                         "mean": d.get("mean"), "sd": d.get("std"), "median": d.get("q2"),
                         "min": d.get("min"), "max": d.get("max")})
    return pd.DataFrame(rows)


def category_counts(result, variable):
    """Counts and percentages of the levels of a categorical variable from a describe result."""
    lab = labels_of(variable)
    rows = []
    for r in rows_for(result, variable):
        counts = (r.get("data") or {}).get("counts") or {}
        total = sum(counts.values())
        for c, n in counts.items():
            rows.append({"variable": variable.label, "level": lab.get(str(c), str(c)), "count": n,
                         "percent": 100 * n / total if total else np.nan})
    return pd.DataFrame(rows)


def describe_by(variables, group, base_filter=None):
    """Describe the variables separately in each level of `group` (one federated call per level).
    Returns (numeric table, categorical table) with a column per group level."""
    num, cat = [], []
    for c, label in labels_of(group).items():
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis(variables, f).describe()
        n = numeric_summary(res, [v for v in variables if v.is_numerical()])
        if len(n):
            num.append(n.assign(group=label))
        for v in variables:
            if v.is_categorical():
                k = category_counts(res, v)
                if len(k):
                    cat.append(k.assign(group=label))
    num = pd.concat(num, ignore_index=True) if num else pd.DataFrame()
    cat = pd.concat(cat, ignore_index=True) if cat else pd.DataFrame()
    if len(num):
        num = num.pivot(index="variable", columns="group", values=["n", "mean", "sd"])
    if len(cat):
        cat = cat.pivot(index=["variable", "level"], columns="group", values="percent")
    return num, cat


def crosstab(exposure, outcome, base_filter=None, event="1"):
    """Cross-tabulation of outcome by exposure, as the MIP does it: counts come from one
    describe per exposure level, the p-value from the Chi-squared Test on the whole cohort."""
    lab_x, lab_y = labels_of(exposure), labels_of(outcome)
    rows = []
    for c, label in lab_x.items():
        f = (F(exposure) == c) & F(outcome).is_not_null()
        if base_filter is not None:
            f = f & base_filter
        res = analysis([exposure, outcome], f).describe()
        counts = {}
        for r in rows_for(res, outcome):
            counts = (r.get("data") or {}).get("counts") or {}
        n = sum(counts.values())
        if n == 0:
            continue
        row = {exposure.label: label, "n": n}
        for yc, ylabel in lab_y.items():
            row[ylabel] = counts.get(yc, 0)
        row["% " + lab_y.get(event, event)] = 100 * counts.get(event, 0) / n
        rows.append(row)
    table = pd.DataFrame(rows)
    f = F(exposure).is_not_null() & F(outcome).is_not_null()
    if base_filter is not None:
        f = f & base_filter
    chi = analysis([exposure, outcome], f, drop_missing=True).chi_square_test(x=exposure, y=outcome)
    stats = chi.raw if isinstance(chi.raw, dict) else {}
    table.attrs["chi2"] = stats.get("chi2")
    table.attrs["p"] = stats.get("p_value", stats.get("p"))
    table.attrs["outcome"] = outcome.label
    return table


def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def show_crosstab(table, title=None):
    """Display a crosstab table, its chi-square test and a bar chart of the percentages."""
    pct_col = [c for c in table.columns if c.startswith("% ")][0]
    display(table.style.format({pct_col: "{:.1f}"}).hide(axis="index"))
    print(f"Chi-squared Test: chi2 = {table.attrs['chi2']:.2f}, p = {fmt_p(table.attrs['p'])}")
    ax = table.set_index(table.columns[0])[pct_col].plot(kind="bar", color="#4C72B0", figsize=(5, 3))
    ax.set_ylabel(pct_col)
    ax.set_title(title or f"{table.attrs['outcome']} by {table.columns[0]}")
    ax.set_xlabel("")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


def odds_ratio_table(result, variables):
    """Logistic regression result -> one row per term with odds ratio, 95% CI and p.
    Categorical covariates appear as 'variable = level' relative to the reference level
    (the first level of the variable) that the platform leaves out."""
    by_code = {v._code: v for v in variables}

    def pretty(term):
        term = str(term)
        if "[" in term and term.endswith("]"):
            base, level = term[:-1].split("[", 1)
            v = by_code.get(base)
            return f"{v.label} = {labels_of(v).get(level, level)}" if v else term
        v = by_code.get(term)
        return f"{v.label} (per unit)" if v else term

    frame = result.to_frame().copy()
    frame = frame[~frame["feature"].astype(str).str.lower().isin(["intercept", "const"])]
    frame["term"] = frame["feature"].map(pretty)
    for col in ("odds_ratio", "or_lower_ci", "or_upper_ci", "p"):
        if col not in frame.columns:
            frame[col] = np.nan
    out = frame[["term", "odds_ratio", "or_lower_ci", "or_upper_ci", "p"]]
    return out.rename(columns={"or_lower_ci": "ci_low", "or_upper_ci": "ci_high"}).reset_index(drop=True)


def show_odds_ratios(table, title):
    """Table and forest plot (log scale) of odds ratios."""
    display(table.style.format({"odds_ratio": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": "{:.4f}"}).hide(axis="index"))
    t = table.iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(t) + 1.5))
    ax.errorbar(t["odds_ratio"], range(len(t)),
                xerr=[t["odds_ratio"] - t["ci_low"], t["ci_high"] - t["odds_ratio"]], fmt="o", capsize=4)
    ax.axvline(1, linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel("odds ratio (log scale)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


print("Toolbox ready.")

## 3. The variables of this scenario

| Code | Label in the MIP | Tree location (under IST) | Role | Values |
|---|---|---|---|---|
| `RDEF1` | Face deficit | Clinical assessment at admission / Neurological deficits | F of FAST | 0 No; 1 Yes; 2 Cannot assess |
| `RDEF2` | Arm/hand deficit | Clinical assessment at admission / Neurological deficits | A of FAST | 0 No; 1 Yes; 2 Cannot assess |
| `RDEF3` | Leg/foot deficit | Clinical assessment at admission / Neurological deficits | sign not in FAST | 0 No; 1 Yes; 2 Cannot assess |
| `RDEF4` | Dysphasia | Clinical assessment at admission / Neurological deficits | S of FAST (language) | 0 No; 1 Yes; 2 Cannot assess |
| `RDEF5` | Hemianopia | Clinical assessment at admission / Neurological deficits | E of BE-FAST | 0 No; 1 Yes; 2 Cannot assess |
| `RDEF6` | Visuospatial disorder | Clinical assessment at admission / Neurological deficits | sign not in FAST | 0 No; 1 Yes; 2 Cannot assess |
| `RDEF7` | Brainstem/cerebellar signs | Clinical assessment at admission / Neurological deficits | B of BE-FAST | 0 No; 1 Yes; 2 Cannot assess |
| `RDEF8` | Other deficit | Clinical assessment at admission / Neurological deficits | sign not in FAST | 0 No; 1 Yes; 2 Cannot assess |
| `STYPE` | OCSP stroke syndrome | Clinical assessment at admission / Stroke syndrome | where in the brain | 1 TACS - total anterior circulation syndrome; 2 PACS - partial anterior circulation syndrome; 3 POCS - posterior circulation syndrome; 4 LACS - lacunar syndrome; 5 Other |
| `OCCODE` | Six-month outcome | Follow-up at 6 months / Functional outcome | what happened six months later | 1 Dead; 2 Dependent; 3 Not recovered (independent); 4 Fully recovered |
| `RDELAY` | Onset to randomisation delay | Demographics and event / Event | T of FAST: hours from first symptoms to entering the trial | integer, 0-48 hours |
| `RDELAYGRP` | Onset to randomisation delay group | Demographics and event / Event | the same delay in bands; curation-derived | 1 0-3 hours; 2 >3-6 hours; 3 >6-12 hours; 4 >12-24 hours; 5 >24-48 hours |
| `RSLEEP` | Symptoms noted on waking | Demographics and event / Event | wake-up stroke | 0 No; 1 Yes |
| `FDIAG` | Final diagnosis of initial event | Stroke characteristics | discussion: patients who did not have a stroke; curation-derived | 1 Ischaemic stroke; 2 Haemorrhagic stroke; 3 Stroke of unknown type; 4 Not a stroke; 9 Unknown |
| `TD` | Time to death or censoring | Death and survival / Survival analysis | optional Step 5: day of death | integer, 0-1400 days |
| `DIED` | Died | Death and survival / Survival analysis | optional Step 5: death | 0 No; 1 Yes |

**Words you will meet.** A *sign*, or deficit, is something the doctor finds wrong, such as a
weak arm; each is coded No, Yes or Cannot assess (for example when a drowsy patient cannot do a
vision test). *Dysphasia* is difficulty finding or understanding words: it is how IST recorded
the S of FAST, as slurred speech was not recorded separately. *Hemianopia*, the loss of half of
the visual field, is the E of BE-FAST; *brainstem/cerebellar signs* (balance, coordination,
double vision, swallowing) are the B. The *stroke syndrome* says where the stroke is: TACS, a
large stroke in the front and middle of one half of the brain; PACS, a smaller one in the same
area; LACS, a small deep stroke causing pure weakness or numbness; POCS, a stroke at the back of
the brain or in the brainstem. *Dependent* means needing help from another person for everyday
activities.

In [ ]:
face     = dm.variables["Face deficit"]
arm      = dm.variables["Arm/hand deficit"]
leg      = dm.variables["Leg/foot deficit"]
speech   = dm.variables["Dysphasia"]
eyes     = dm.variables["Hemianopia"]
spatial  = dm.variables["Visuospatial disorder"]
balance  = dm.variables["Brainstem/cerebellar signs"]
other    = dm.variables["Other deficit"]
stype    = dm.variables["OCSP stroke syndrome"]
occode   = dm.variables["Six-month outcome"]
delay    = dm.variables["Onset to randomisation delay"]
delaygrp = dm.variables["Onset to randomisation delay group"]
wakeup   = dm.variables["Symptoms noted on waking"]
td       = dm.variables["Time to death or censoring"]
died     = dm.variables["Died"]

display(mip.to_frame([face, arm, leg, speech, eyes, spatial, balance, other, stype, occode,
                      delay, delaygrp, wakeup, td, died]))

## Step 1. What does a stroke look like?

One `describe()` of the eight signs, with no filter: every patient. The chart shows the
percentage of patients with each sign: the FAST signs in red, the two signs BE-FAST adds in
orange.

*Expected:* arm or hand 85.6%, leg or foot 75.5%, face 72.5%, dysphasia 43.8%, visuospatial
disorder 16.4%, hemianopia 15.9%, brainstem/cerebellar signs 11.0%, other 6.3%. Vision and
spatial awareness could not be tested in about one patient in five, usually because the patient
was too drowsy.

Weakness of one side of the body is the hallmark of a stroke: more than 8 in 10 patients had a
weak arm or hand, 7 in 10 a weak face, and 4 in 10 had difficulty with words.

In [ ]:
signs = {"Face": face, "Arm/hand": arm, "Leg/foot": leg, "Speech (dysphasia)": speech,
         "Eyes (hemianopia)": eyes, "Visuospatial": spatial, "Balance (brainstem/cerebellar)": balance,
         "Other": other}
profile = analysis(list(signs.values())).describe()

rows = []
for name, v in signs.items():
    pct = category_counts(profile, v).set_index("level")["percent"]
    rows.append({"sign": name, "% yes": pct.get("Yes", 0), "% cannot assess": pct.get("Cannot assess", 0)})
signs_table = pd.DataFrame(rows).sort_values("% yes")
display(signs_table.iloc[::-1].style.format({"% yes": "{:.1f}", "% cannot assess": "{:.1f}"}).hide(axis="index"))

# FAST signs in red, the two signs BE-FAST adds in orange, the others in grey
colour = {"Face": "#C44E52", "Arm/hand": "#C44E52", "Speech (dysphasia)": "#C44E52",
          "Eyes (hemianopia)": "#DD8452", "Balance (brainstem/cerebellar)": "#DD8452"}
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(signs_table["sign"], signs_table["% yes"], color=[colour.get(s, "#8C8C8C") for s in signs_table["sign"]])
for y, value in enumerate(signs_table["% yes"]):
    ax.text(value + 1, y, f"{value:.0f}%", va="center")             # the percentage at the end of each bar
ax.set_xlim(0, 100)
ax.set_xlabel("% of patients with the sign")
ax.set_title("What a stroke looks like. Red: FAST signs; orange: added by BE-FAST")
plt.tight_layout()
plt.show()

## Step 2. Would FAST have recognised it?

A stroke counts as **missed** by FAST when the doctor examined the face, the arm and speech and
found all three normal: the filter keeps the patients with code 0 (No) for all three. For
BE-FAST, the eyes and balance must be normal too. For everyone, for the strokes FAST would miss
and for those BE-FAST would miss, one `describe()` gives the stroke syndrome and the outcome six
months later.

*Expected:* FAST would miss 938 patients (4.8%, about one in twenty) and BE-FAST 254 (1.3%,
about one in seventy-five). The 100 squares show it at a glance.

In [ ]:
missed_by_fast = (F(face) == "0") & (F(arm) == "0") & (F(speech) == "0")
missed_by_befast = missed_by_fast & (F(balance) == "0") & (F(eyes) == "0")

everyone = analysis([stype, occode]).describe()
fast_missed = analysis([stype, occode], missed_by_fast).describe()
befast_missed = analysis([stype, occode], missed_by_befast).describe()

n_all = category_counts(everyone, stype)["count"].sum()
n_fast = category_counts(fast_missed, stype)["count"].sum()
n_befast = category_counts(befast_missed, stype)["count"].sum()
print(f"All patients:      {n_all:,}")
print(f"Missed by FAST:    {n_fast:,} ({100 * n_fast / n_all:.1f}%)")
print(f"Missed by BE-FAST: {n_befast:,} ({100 * n_befast / n_all:.1f}%)")

# the same result as 100 squares, one square for each 1% of the patients
only_befast = round(100 * (n_fast - n_befast) / n_all)
both_miss = round(100 * n_befast / n_all)
squares = {"FAST would spot it": 100 - only_befast - both_miss,
           "only BE-FAST would spot it": only_befast, "both would miss it": both_miss}
colours = ["#55A868", "#DD8452", "#C44E52"]
cells = [c for c, n in zip(colours, squares.values()) for _ in range(n)]
fig, ax = plt.subplots(figsize=(8, 3))
for i, c in enumerate(cells):
    ax.add_patch(plt.Rectangle((i % 20, i // 20), 0.85, 0.85, color=c))
ax.set_xlim(0, 20)
ax.set_ylim(5, 0)                                       # first row at the top
ax.set_aspect("equal")
ax.axis("off")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in colours],
          labels=[f"{label}: {n}" for label, n in squares.items()],
          loc="upper center", bbox_to_anchor=(0.5, 0), ncol=3, frameon=False)
ax.set_title("Out of 100 stroke patients in IST")
plt.tight_layout()
plt.show()

### Where are the strokes FAST misses, and what happened to those patients?

*Expected:* in everyone, 11.5% of the strokes are at the back of the brain (POCS); among the
strokes FAST misses, 65.9%. FAST misses 618 of the 2,228 POCS strokes, more than one in four,
because they show up as loss of balance, dizziness or loss of vision rather than as a weak face
or arm. In IST, BE-FAST catches every one of them, partly by definition, since doctors
classified POCS from these same signs. What BE-FAST still misses is a smaller stroke at the
front (PACS, for example weakness of a leg only) or an unclassified one ("Other", 57 patients).

Six months later, 35.4% of the patients whose stroke FAST would miss were dead or dependent,
against 62.9% of everyone: their strokes were milder on average, but one in three is not
harmless.

In [ ]:
def shares(result, variable):
    """Percentage of each level of `variable` in a describe result, in the order of its codes."""
    pct = category_counts(result, variable).set_index("level")["percent"]
    return pct.reindex(list(labels_of(variable).values())).fillna(0).rename_axis(None)


groups = {"everyone": everyone, "missed by FAST": fast_missed, "missed by BE-FAST": befast_missed}
where = pd.DataFrame({name: shares(res, stype) for name, res in groups.items()}).T
where.columns = [c.split(" - ")[0] for c in where.columns]        # TACS, PACS, POCS, LACS, Other
later = pd.DataFrame({name: shares(res, occode) for name, res in groups.items()}).T
display(where.style.format("{:.1f}"))
display(later.style.format("{:.1f}"))

fig, axes = plt.subplots(1, 2, figsize=(14, 3.4), sharey=True)
where.plot(kind="barh", stacked=True, ax=axes[0], color=["#4C72B0", "#8EBAD9", "#C44E52", "#8172B2", "#8C8C8C"])
later.plot(kind="barh", stacked=True, ax=axes[1], color=["#8B0000", "#DD8452", "#8EBAD9", "#55A868"])
axes[0].set_title("Where in the brain? Stroke syndrome")
axes[1].set_title("Six months later")
axes[0].invert_yaxis()                                  # shared axis: everyone on top in both panels
for ax in axes:
    ax.set_xlabel("% of the group")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=3, frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

## Step 3. T is for Time

A histogram of the delay between the first symptoms and entry into the trial, one bar per hour
(47 bins from 1 to 48 hours; the last bar holds 47 and 48). The green band is today's window for
a clot-dissolving drug, 4.5 hours.

*Expected:* 1,562 patients (8.0%) entered within 4 hours and a third after more than 24 hours;
median delay 19 hours; spikes at 6, 12, 24 and 36 hours; 29.3% noticed their symptoms on waking.

- In the 1990s there was no emergency treatment for stroke, so nobody rushed. Today a
  clot-dissolving drug (thrombolysis) can be given up to 4.5 hours after the start, and a clot
  can be pulled out of a large artery (thrombectomy) up to 24 hours in selected patients. That
  is the T of FAST: call the emergency number at once. The delay here counts to entry in the
  trial, after the doctor's examination, so arrival in hospital came somewhat earlier.
- The spikes are not biology. When the exact time was not known, the doctor wrote a round
  number. Real data carry the fingerprints of how they were collected.
- A stroke noticed on waking began during sleep, and nobody knows exactly when: this still
  complicates treatment decisions today.

In [ ]:
hist = analysis([delay], drop_missing=True).histogram(variable=delay, bins=47)
h = hist.raw["histogram"][0]
edges, counts = h["bins"], [c or 0 for c in h["counts"]]       # 47 bins from 1 to 48 hours: one bar per hour

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.axvspan(0, 4.5, color="#55A868", alpha=0.3, label="today's window for a clot-dissolving drug: 4.5 hours")
ax.bar(edges[:-1], counts, width=0.85, align="edge", color="#4C72B0")
ax.set_xlim(0, 49)
ax.set_xticks(range(0, 49, 6))
ax.set_xlabel("hours from the first symptoms to entering the trial")
ax.set_ylabel("patients")
ax.set_title("T is for Time: how late patients came, 1991-1996")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()
print(f"Entered the trial within 4 hours: {sum(counts[:4]):,} patients ({100 * sum(counts[:4]) / sum(counts):.1f}%)")

timing = analysis([delay, delaygrp, wakeup]).describe()
display(numeric_summary(timing, [delay]).style.format({"mean": "{:.1f}", "sd": "{:.1f}"}).hide(axis="index"))
bands = category_counts(timing, delaygrp).set_index("level").reindex(list(labels_of(delaygrp).values()))
display(bands[["count", "percent"]].style.format({"percent": "{:.1f}"}))
display(category_counts(timing, wakeup).style.format({"percent": "{:.1f}"}).hide(axis="index"))

## Step 4. Six months later

The six-month outcome of each kind of stroke, as shares of the patients with a known outcome:
dead, dependent, independent but not recovered, fully recovered. The 55 "Other" patients are
left out: too few.

*Expected:* dead 40.4% after a TACS, 20.0% after a PACS, 18.0% after a POCS and 9.1% after a
LACS; fully recovered 5.0%, 18.2%, 24.1% and 23.8%.

The size and place of a stroke, judged at the bedside from the signs, foretell what happens
next. After a large stroke at the front of the brain, 4 in 10 patients had died within six
months and only 1 in 20 had fully recovered. After a small deep stroke, fewer than 1 in 10 had
died and about 1 in 4 had fully recovered. Strokes at the back of the brain, the ones FAST
misses, are not harmless either: almost 1 in 5 of those patients died.

In [ ]:
table = crosstab(stype, occode)                                   # counts of each outcome, by kind of stroke
outcomes = list(labels_of(occode).values())                       # Dead, Dependent, Not recovered, Fully recovered
share = table.set_index("OCSP stroke syndrome")[outcomes]
share = share.drop(index=[s for s in share.index if s.startswith("Other")])     # 55 patients: too few
share = share.div(share.sum(axis=1), axis=0).mul(100)
share.index = [s.replace(" - ", ": ") for s in share.index]
display(share.style.format("{:.1f}"))

ax = share.plot(kind="barh", stacked=True, figsize=(10, 3.8), color=["#8B0000", "#DD8452", "#8EBAD9", "#55A868"])
ax.invert_yaxis()                                                 # TACS on top
ax.set_xlabel("% of patients six months after the stroke")
ax.set_ylabel("")
ax.set_title("Six months later, by the kind of stroke seen at admission")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=4, frameon=False)
plt.tight_layout()
plt.show()

## Step 5 (if time allows). When do the deaths happen?

For each kind of stroke, a histogram of the day of death over the first six months (6 bins,
about one month each), added up from the left and divided by the number of patients with that
kind of stroke (from Step 2).

*Expected:* dead by the end of the first month about 26% (TACS), 11% (PACS), 11% (POCS) and 4%
(LACS); by six months 40%, 20%, 18% and 9%. Most deaths come in the first weeks, and the gap
between a large and a small stroke opens immediately: care in the first days matters most.

In [ ]:
size = category_counts(everyone, stype).set_index("level")["count"]      # patients of each kind (Step 2)

fig, ax = plt.subplots(figsize=(8, 4.2))
for level, label in labels_of(stype).items():
    if label.startswith("Other"):
        continue                                                        # 57 patients: too few
    deaths = analysis([td], (F(died) == "1") & (F(td) <= 182) & (F(stype) == level),
                      drop_missing=True).histogram(variable=td, bins=6)
    h = deaths.raw["histogram"][0]
    dead_pct = 100 * np.cumsum([c or 0 for c in h["counts"]]) / size[label]
    ax.plot(h["bins"], [0] + list(dead_pct), marker="o", label=label.split(" - ")[0])
ax.set_xlabel("days after entering the trial")
ax.set_ylabel("% of the group dead")
ax.set_title("When do the deaths happen? By kind of stroke")
ax.legend()
plt.tight_layout()
plt.show()

## What to remember

- More than 8 in 10 stroke patients have a weak arm, and FAST signs are present in about 95 in
  100.
- FAST misses about 1 in 20 strokes, mostly at the back of the brain; Balance and Eyes catch
  most of them. Missed strokes are milder on average, but not harmless.
- In the 1990s patients reached the trial a day after their stroke; today, minutes matter.
- The kind of stroke seen at admission already says a lot about the outcome six months later.

**Questions for discussion**

1. FAST misses about 1 in 20 strokes, BE-FAST about 1 in 75. Why might a campaign still teach
   FAST rather than BE-FAST?
2. IST patients were examined by doctors after a diagnosis of stroke. Would FAST work better or
   worse in the hands of a relative at home?
3. Among the 938 patients without a face, arm or speech sign, 58 (6.2%) turned out not to have
   had a stroke at all, against 2.2% in the whole trial (`Final diagnosis of initial event`,
   level "Not a stroke"). What does this say about recognising a stroke from signs alone?
4. Why does the delay histogram have spikes at round numbers? Where else in medical data would
   you expect such rounding?